In [1]:
import json
import importlib
from dataclasses import dataclass, field
from datetime import datetime, date
from typing import Any, Dict, Optional

import fetch_retry
from fetch_retry import fetch_com_retry

In [2]:

@dataclass
class PncpApiConfig:
    """
    Responsável por armazenar a configuração base da API do PNCP.
    """
    base_url: str = "https://pncp.gov.br/api/consulta"
    endpoint: str = "/v1/contratacoes/proposta"

    @property
    def url_final(self) -> str:
        return f"{self.base_url}{self.endpoint}"


In [3]:
@dataclass
class ConsultaPropostaParams:
    """
    Responsável por representar os parâmetros da consulta.
    """
    codigo_modalidade: int = 6
    pagina: int = 1
    tamanho_pagina: int = 50
    data_final: int = field(default_factory=lambda: int(datetime.now().strftime("%Y%m%d")))

    def to_dict(self) -> Dict[str, Any]:
        return {
            "dataFinal": self.data_final,
            "codigoModalidadeContratacao": self.codigo_modalidade,
            "pagina": self.pagina,
            "tamanhoPagina": self.tamanho_pagina,
        }

In [4]:

class ModuloFetcherLoader:
    """
    Responsável por recarregar dinamicamente o módulo de fetch,
    útil em ambiente de desenvolvimento/notebook.
    """

    def __init__(self, module_name: str = "fetch_retry") -> None:
        self.module_name = module_name

    def reload(self) -> None:
        module = importlib.import_module(self.module_name)
        importlib.reload(module)
        print(f"Módulo '{self.module_name}' recarregado com sucesso.")

In [5]:
class JsonFormatter:
    """
    Responsável por formatar estruturas Python em JSON legível.
    """

    @staticmethod
    def format(data: Any) -> str:
        return json.dumps(data, indent=4, ensure_ascii=False)

In [6]:
class PncpConsultaService:
    """
    Responsável por executar a consulta na API do PNCP.
    """

    def __init__(self, api_config: PncpApiConfig) -> None:
        self.api_config = api_config

    def consultar_propostas(self, params: ConsultaPropostaParams) -> Dict[str, Any]:
        return fetch_com_retry(self.api_config.url_final, params.to_dict())


In [7]:
class PncpConsultaOrchestrator:
    """
    Responsável por orquestrar o fluxo completo:
    - recarregar módulo de fetch
    - montar parâmetros
    - executar consulta
    - exibir resultados
    """

    def __init__(
        self,
        consulta_service: PncpConsultaService,
        formatter: JsonFormatter,
        loader: Optional[ModuloFetcherLoader] = None,
    ) -> None:
        self.consulta_service = consulta_service
        self.formatter = formatter
        self.loader = loader
        self.data_hoje = date.today().strftime("%Y-%m-%d")

    def executar(self) -> Dict[str, Any]:
        if self.loader:
            self.loader.reload()

        params = ConsultaPropostaParams()

        print(f"Data final da consulta: {params.data_final}")
        print(f"Data de execução do pipeline: {self.data_hoje}")
        print(f"URL da consulta: {self.consulta_service.api_config.url_final}")
        print("Parâmetros enviados:")
        print(self.formatter.format(params.to_dict()))

        resposta = self.consulta_service.consultar_propostas(params)

        print("\nResposta bruta:")
        print(resposta)

        print("\nResposta formatada:")
        print(self.formatter.format(resposta))

        return resposta

In [8]:

if __name__ == "__main__":
    api_config = PncpApiConfig()
    consulta_service = PncpConsultaService(api_config)
    formatter = JsonFormatter()
    loader = ModuloFetcherLoader()

    orchestrator = PncpConsultaOrchestrator(
        consulta_service=consulta_service,
        formatter=formatter,
        loader=loader,
    )

    resultado = orchestrator.executar()

Módulo 'fetch_retry' recarregado com sucesso.
Data final da consulta: 20260319
Data de execução do pipeline: 2026-03-19
URL da consulta: https://pncp.gov.br/api/consulta/v1/contratacoes/proposta
Parâmetros enviados:
{
    "dataFinal": 20260319,
    "codigoModalidadeContratacao": 6,
    "pagina": 1,
    "tamanhoPagina": 50
}
Código de resposta 200 (OK)

Resposta bruta:
([{'dataAtualizacao': '2025-10-14T10:08:45', 'orgaoEntidade': {'cnpj': '30066661000180', 'razaoSocial': 'SUPERINTENDENCIA MUNICIPAL DE TRANSITO', 'poderId': 'E', 'esferaId': 'M'}, 'anoCompra': 2025, 'sequencialCompra': 15, 'numeroCompra': '43248', 'processo': '2025017256', 'objetoCompra': 'ADESÃO Nº 038/2025\nADESÃO ATA DE REGISTRO DE PREÇOS Nº 067/2025 DE 19/03/2025 EM DECORRÊNCIA PREGÃO ELETRONICO/SRP Nº 004/2025 - PROCESSO ADMINISTRATIVO Nº 2024013537 DA PREFEITURA MUNICIPAL DE CALDAS NOVAS/GO, NA CONDIÇÃO DE ?CARONA?, PARA EVENTUAL AQUISIÇÃO DE MATERIAL DE CONSTRUÇÃO EM ATENDIMENTO ÀS NECESSIDADES DA SUPERINTENDÊNCIA 

In [57]:
import json
import math
import os
import re
import unicodedata
from collections import Counter
from datetime import datetime
from typing import Any, Dict, List, Optional

from fetch_retry import fetch_com_retry


class PncpPaginacaoParaJson:
    """
    Responsável por:
    - percorrer páginas da API PNCP
    - coletar múltiplas modalidades
    - remover duplicados
    - normalizar os registros
    - enriquecer os dados com categoria e palavras-chave
    - organizar os dados por modalidade
    - gerar resumo por UF e município
    - salvar JSON premium, limpo e profissional
    """

    def __init__(
        self,
        base_url: str = "https://pncp.gov.br/api/consulta",
        endpoint: str = "/v1/contratacoes/proposta",
        codigo_modalidade: Optional[int] = None,
        tamanho_pagina: int = 50,
        data_final: Optional[int] = None,
        pagina_inicial: int = 1,
        max_paginas: Optional[int] = None,
    ) -> None:
        self.base_url = base_url
        self.endpoint = endpoint
        self.url_final = f"{self.base_url}{self.endpoint}"
        self.codigo_modalidade = codigo_modalidade
        self.tamanho_pagina = tamanho_pagina
        self.data_final = data_final or int(datetime.now().strftime("%Y%m%d"))
        self.pagina_inicial = pagina_inicial
        self.max_paginas = max_paginas

        self.stopwords = {
            "a", "o", "e", "de", "da", "do", "das", "dos", "para", "por", "com",
            "sem", "em", "na", "no", "nas", "nos", "um", "uma", "uns", "umas",
            "ao", "aos", "à", "às", "que", "se", "sob", "sobre", "entre", "ou",
            "como", "ser", "será", "sendo", "visando", "atender", "necessidades",
            "necessidade", "destinados", "destinadas", "destinado", "destinada",
            "conforme", "termo", "referencia", "edital", "anexo", "anexos",
            "prestacao", "servicos", "servico", "contratacao", "empresa",
            "aquisição", "aquisicao", "objeto", "municipio", "municipal"
        }

    def _montar_params(self, pagina: int, codigo_modalidade: int) -> Dict[str, Any]:
        return {
            "dataFinal": self.data_final,
            "codigoModalidadeContratacao": codigo_modalidade,
            "pagina": pagina,
            "tamanhoPagina": self.tamanho_pagina,
        }

    def _extrair_registros_da_resposta(self, resposta: Any) -> List[Dict[str, Any]]:
        if isinstance(resposta, list):
            return resposta

        if isinstance(resposta, tuple):
            if len(resposta) > 0 and isinstance(resposta[0], list):
                return resposta[0]

        raise TypeError(
            f"Formato inesperado retornado por fetch_com_retry: "
            f"{type(resposta)} | valor: {repr(resposta)[:500]}"
        )

    def _safe_float(self, valor: Any) -> Optional[float]:
        try:
            if valor is None or valor == "":
                return None
            return float(valor)
        except (TypeError, ValueError):
            return None

    def _valor_para_json(self, valor: Any) -> Any:
        if isinstance(valor, float):
            if math.isnan(valor) or math.isinf(valor):
                return None
        return valor

    def _remover_campos_vazios(self, estrutura: Any) -> Any:
        if isinstance(estrutura, dict):
            novo = {}
            for chave, valor in estrutura.items():
                valor_limpo = self._remover_campos_vazios(valor)

                if valor_limpo is None:
                    continue
                if valor_limpo == "":
                    continue
                if valor_limpo == []:
                    continue
                if valor_limpo == {}:
                    continue

                novo[chave] = valor_limpo
            return novo

        if isinstance(estrutura, list):
            nova_lista = []
            for item in estrutura:
                item_limpo = self._remover_campos_vazios(item)
                if item_limpo is None:
                    continue
                if item_limpo == "":
                    continue
                if item_limpo == []:
                    continue
                if item_limpo == {}:
                    continue
                nova_lista.append(item_limpo)
            return nova_lista

        return self._valor_para_json(estrutura)

    def _normalizar_texto(self, texto: str) -> str:
        if not texto:
            return ""
        texto = texto.lower().strip()
        texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
        return texto

    def _tokenizar(self, texto: str) -> List[str]:
        texto = self._normalizar_texto(texto)
        tokens = re.findall(r"[a-zA-Z0-9]+", texto)
        return [t for t in tokens if len(t) > 2 and t not in self.stopwords]

    def _extrair_palavras_chave(self, texto: str, limite: int = 8) -> List[str]:
        tokens = self._tokenizar(texto)
        frequencia = Counter(tokens)
        palavras = [palavra for palavra, _ in frequencia.most_common(limite)]
        return palavras

    def _classificar_categoria_objeto(self, texto: str) -> str:
        t = self._normalizar_texto(texto)

        categorias = {
            "tecnologia": [
                "software", "sistema", "informatica", "internet", "rede", "computador",
                "notebook", "scanner", "licenca", "antivirus", "camera", "videomonitoramento",
                "monitoramento", "equipamentos de informatica", "fibra optica"
            ],
            "engenharia_obras": [
                "obra", "construcao", "reforma", "pavimentacao", "drenagem", "engenharia",
                "reservatorio", "ubs", "unidade basica", "canalizacao", "adutora"
            ],
            "saude": [
                "saude", "hospital", "medico", "medicos", "consulta", "cirurgia", "exames",
                "mamografia", "ambulancia", "pacientes", "piscina", "psiquiatria"
            ],
            "educacao": [
                "escola", "creche", "merenda", "educacao", "alunos", "unidades escolares"
            ],
            "alimentacao": [
                "alimentos", "generos alimenticios", "panificados", "refeicoes", "marmitex",
                "peixes", "cestas basicas", "ovos de pascoa", "buffet"
            ],
            "veiculos_logistica": [
                "veiculo", "veiculos", "frota", "transporte", "caminhao", "van",
                "moto niveladora", "seguro veicular", "caçambas", "cacambas"
            ],
            "manutencao_materiais": [
                "material", "materiais", "eletrico", "eletricos", "madeira", "mobilia",
                "suprimentos", "insumos", "quimicos", "uniforme", "vestuario"
            ],
            "servicos_gerais": [
                "prestacao de servicos", "servicos", "assessoria", "consultoria",
                "publicidade", "comunicacao", "pesquisa", "limpeza", "manutencao"
            ],
        }

        for categoria, termos in categorias.items():
            if any(termo in t for termo in termos):
                return categoria

        return "outros"

    def _parse_data(self, valor: Optional[str]) -> str:
        if not valor:
            return ""
        return valor

    def buscar_paginas_de_uma_modalidade(self, codigo_modalidade: int) -> Dict[str, Any]:
        pagina = self.pagina_inicial
        registros_modalidade: List[Dict[str, Any]] = []
        paginas_processadas = 0

        while True:
            if self.max_paginas is not None and paginas_processadas >= self.max_paginas:
                print(f"Limite de páginas atingido para modalidade {codigo_modalidade}: {self.max_paginas}")
                break

            params = self._montar_params(pagina, codigo_modalidade)

            print(f"\nConsultando modalidade {codigo_modalidade} | página {pagina}...")

            resposta_bruta = fetch_com_retry(self.url_final, params)
            registros = self._extrair_registros_da_resposta(resposta_bruta)

            if not registros:
                print(f"Nenhum registro encontrado na página {pagina} da modalidade {codigo_modalidade}.")
                break

            quantidade = len(registros)
            registros_modalidade.extend(registros)
            paginas_processadas += 1

            print(f"Modalidade atual: {codigo_modalidade}")
            print(f"Página atual: {pagina}")
            print(f"Registros retornados nesta página: {quantidade}")
            print(f"Total acumulado da modalidade: {len(registros_modalidade)}")

            if quantidade < self.tamanho_pagina:
                print(f"Última página detectada para modalidade {codigo_modalidade}: {pagina}")
                break

            pagina += 1

        print(f"\nColeta finalizada para modalidade {codigo_modalidade}.")
        print(f"Total coletado na modalidade {codigo_modalidade}: {len(registros_modalidade)}")
        print(f"Total de páginas processadas na modalidade {codigo_modalidade}: {paginas_processadas}")

        return {
            "codigo_modalidade": codigo_modalidade,
            "registros": registros_modalidade,
            "paginas_processadas": paginas_processadas,
        }

    def remover_duplicados(self, dados: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        vistos = set()
        unicos = []
        duplicados = 0

        for item in dados:
            chave = item.get("numeroControlePNCP")

            if chave:
                if chave not in vistos:
                    vistos.add(chave)
                    unicos.append(item)
                else:
                    duplicados += 1
            else:
                unicos.append(item)

        print(f"Total antes da deduplicação: {len(dados)}")
        print(f"Duplicados removidos: {duplicados}")
        print(f"Total após deduplicação: {len(unicos)}")

        return unicos

    def carregar_json_existente(self, caminho_arquivo: str) -> List[Dict[str, Any]]:
        if not os.path.exists(caminho_arquivo):
            print("Arquivo JSON ainda não existe. Será criado um novo.")
            return []

        try:
            with open(caminho_arquivo, "r", encoding="utf-8") as f:
                conteudo = json.load(f)

            if isinstance(conteudo, dict):
                if "dados" in conteudo and isinstance(conteudo["dados"], list):
                    print(f"JSON existente carregado com {len(conteudo['dados'])} registros.")
                    return conteudo["dados"]

                if "modalidades" in conteudo and isinstance(conteudo["modalidades"], dict):
                    dados = []
                    for bloco in conteudo["modalidades"].values():
                        if isinstance(bloco, dict) and isinstance(bloco.get("dados"), list):
                            dados.extend(bloco["dados"])
                    print(f"JSON existente carregado com {len(dados)} registros.")
                    return dados

            if isinstance(conteudo, list):
                print(f"JSON existente carregado com {len(conteudo)} registros.")
                return conteudo

            print("JSON existente em formato inesperado. Será considerado vazio.")
            return []

        except json.JSONDecodeError:
            print("JSON existente está inválido ou vazio. Será considerado vazio.")
            return []

    def normalizar_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        orgao = item.get("orgaoEntidade") or {}
        unidade = item.get("unidadeOrgao") or {}
        amparo = item.get("amparoLegal") or {}

        objeto_compra = item.get("objetoCompra")
        data_publicacao = item.get("dataPublicacaoPncp")
        data_abertura = item.get("dataAberturaProposta")
        data_encerramento = item.get("dataEncerramentoProposta")
        valor_estimado = self._safe_float(item.get("valorTotalEstimado"))
        valor_homologado = self._safe_float(item.get("valorTotalHomologado"))

        registro = {
            "objeto_compra": objeto_compra,
            "numero_controle_pncp": item.get("numeroControlePNCP"),
            "valor_total_estimado": valor_estimado,
            "valor_total_homologado": valor_homologado,
            "informacao_complementar": item.get("informacaoComplementar"),
            "id": item.get("numeroControlePNCP"),
            "categoria_objeto": self._classificar_categoria_objeto(objeto_compra or ""),
            "palavras_chave": self._extrair_palavras_chave(objeto_compra or ""),
            "id": item.get("numeroControlePNCP"),           
            "ano_compra": item.get("anoCompra"),
            "sequencial_compra": item.get("sequencialCompra"),
            "numero_compra": item.get("numeroCompra"),
            "processo": item.get("processo"),           
            "modalidade_id": item.get("modalidadeId"),
            "modalidade_nome": item.get("modalidadeNome"),
            "modo_disputa_id": item.get("modoDisputaId"),
            "modo_disputa_nome": item.get("modoDisputaNome"),
            "tipo_instrumento_convocatorio_codigo": item.get("tipoInstrumentoConvocatorioCodigo"),
            "tipo_instrumento_convocatorio_nome": item.get("tipoInstrumentoConvocatorioNome"),
            "situacao_compra_id": item.get("situacaoCompraId"),
            "situacao_compra_nome": item.get("situacaoCompraNome"),
            "srp": item.get("srp"),            
            "datas": {
                "publicacao_pncp": data_publicacao,
                "abertura_proposta": data_abertura,
                "encerramento_proposta": data_encerramento,
                "atualizacao": item.get("dataAtualizacao"),
                "atualizacao_global": item.get("dataAtualizacaoGlobal"),
                "inclusao": item.get("dataInclusao"),
            },
            "orgao_entidade": {
                "cnpj": orgao.get("cnpj"),
                "razao_social": orgao.get("razaoSocial"),
                "poder_id": orgao.get("poderId"),
                "esfera_id": orgao.get("esferaId"),
            },
            "unidade_orgao": {
                "uf_nome": unidade.get("ufNome"),
                "uf_sigla": unidade.get("ufSigla"),
                "municipio_nome": unidade.get("municipioNome"),
                "nome_unidade": unidade.get("nomeUnidade"),
                "codigo_ibge": unidade.get("codigoIbge"),
                "codigo_unidade": unidade.get("codigoUnidade"),
            },
            "amparo_legal": {
                "codigo": amparo.get("codigo"),
                "nome": amparo.get("nome"),
                "descricao": amparo.get("descricao"),
            },
            "links": {
                "sistema_origem": item.get("linkSistemaOrigem"),
                "processo_eletronico": item.get("linkProcessoEletronico"),
            },
            "justificativa_presencial": item.get("justificativaPresencial"),
            "usuario_nome": item.get("usuarioNome"),
            "fontes_orcamentarias": item.get("fontesOrcamentarias") or [],
        }

        return self._remover_campos_vazios(registro)

    def ordenar_registros(self, dados: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        return sorted(
            dados,
            key=lambda x: (
                self._parse_data(x.get("datas", {}).get("encerramento_proposta")),
                self._parse_data(x.get("datas", {}).get("publicacao_pncp")),
                (x.get("objeto_compra") or "").lower(),
            ),
        )

    def gerar_resumo_modalidades(self, dados: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        contador = Counter()

        for item in dados:
            modalidade_id = item.get("modalidade_id")
            modalidade_nome = item.get("modalidade_nome")

            if modalidade_id is not None and modalidade_nome:
                contador[(modalidade_id, modalidade_nome)] += 1

        resumo = [
            {
                "modalidadeId": modalidade_id,
                "modalidadeNome": modalidade_nome,
                "quantidade": quantidade,
            }
            for (modalidade_id, modalidade_nome), quantidade in sorted(
                contador.items(),
                key=lambda x: x[0][0],
            )
        ]

        return resumo

    def gerar_resumo_ufs(self, dados: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        contador = Counter()

        for item in dados:
            uf = item.get("unidade_orgao", {}).get("uf_sigla")
            if uf:
                contador[uf] += 1

        return [
            {"uf": uf, "quantidade": quantidade}
            for uf, quantidade in sorted(contador.items(), key=lambda x: (-x[1], x[0]))
        ]

    def gerar_resumo_municipios(self, dados: List[Dict[str, Any]], limite: int = 20) -> List[Dict[str, Any]]:
        contador = Counter()

        for item in dados:
            unidade = item.get("unidade_orgao", {})
            municipio = unidade.get("municipio_nome")
            uf = unidade.get("uf_sigla")

            if municipio and uf:
                contador[(municipio, uf)] += 1

        resumo = [
            {"municipio": municipio, "uf": uf, "quantidade": quantidade}
            for (municipio, uf), quantidade in sorted(
                contador.items(),
                key=lambda x: (-x[1], x[0][0], x[0][1]),
            )[:limite]
        ]

        return resumo

    def gerar_resumo_categorias(self, dados: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        contador = Counter()

        for item in dados:
            categoria = item.get("categoria_objeto", "outros")
            contador[categoria] += 1

        return [
            {"categoria": categoria, "quantidade": quantidade}
            for categoria, quantidade in sorted(contador.items(), key=lambda x: (-x[1], x[0]))
        ]

    def agrupar_por_modalidade(self, dados: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
        grupos: Dict[str, Dict[str, Any]] = {}

        for item in dados:
            modalidade_nome = item.get("modalidade_nome") or "Sem modalidade"
            modalidade_id = item.get("modalidade_id")

            if modalidade_nome not in grupos:
                grupos[modalidade_nome] = {
                    "modalidadeId": modalidade_id,
                    "codigo_modalidade": str(modalidade_id) if modalidade_id is not None else None,
                    "quantidade": 0,
                    "dados": [],
                }

            grupos[modalidade_nome]["dados"].append(item)
            grupos[modalidade_nome]["quantidade"] += 1

        for modalidade_nome in grupos:
            grupos[modalidade_nome]["dados"] = self.ordenar_registros(grupos[modalidade_nome]["dados"])

        grupos_ordenados = dict(
            sorted(
                grupos.items(),
                key=lambda x: (
                    x[1]["modalidadeId"] if x[1]["modalidadeId"] is not None else 999999,
                    x[0],
                ),
            )
        )

        return grupos_ordenados

    def gerar_estatisticas_gerais(
        self,
        dados_normalizados: List[Dict[str, Any]],
        modalidades_processadas: List[int],
        total_paginas: int,
    ) -> Dict[str, Any]:
        resumo_modalidades = self.gerar_resumo_modalidades(dados_normalizados)

        valores_estimados = [
            item.get("valor_total_estimado")
            for item in dados_normalizados
            if isinstance(item.get("valor_total_estimado"), (int, float))
        ]

        datas_publicacao = [
            item.get("datas", {}).get("publicacao_pncp")
            for item in dados_normalizados
            if item.get("datas", {}).get("publicacao_pncp")
        ]

        return self._remover_campos_vazios({
            "total_registros": len(dados_normalizados),
            "total_paginas": total_paginas,
            "total_modalidades": len(resumo_modalidades),
            "modalidades_processadas": modalidades_processadas,
            "valor_total_estimado_somado": round(sum(valores_estimados), 2) if valores_estimados else 0.0,
            "valor_total_estimado_medio": round(sum(valores_estimados) / len(valores_estimados), 2)
            if valores_estimados
            else 0.0,
            "menor_data_publicacao": min(datas_publicacao) if datas_publicacao else None,
            "maior_data_publicacao": max(datas_publicacao) if datas_publicacao else None,
        })

    def montar_payload_premium(
        self,
        dados_brutos: List[Dict[str, Any]],
        modalidades_processadas: List[int],
        total_paginas: int,
    ) -> Dict[str, Any]:
        dados_unicos = self.remover_duplicados(dados_brutos)
        dados_normalizados = [self.normalizar_item(item) for item in dados_unicos]
        dados_normalizados = self.ordenar_registros(dados_normalizados)

        resumo_modalidades = self.gerar_resumo_modalidades(dados_normalizados)
        resumo_ufs = self.gerar_resumo_ufs(dados_normalizados)
        resumo_municipios = self.gerar_resumo_municipios(dados_normalizados)
        resumo_categorias = self.gerar_resumo_categorias(dados_normalizados)
        grupos_modalidade = self.agrupar_por_modalidade(dados_normalizados)
        estatisticas = self.gerar_estatisticas_gerais(
            dados_normalizados=dados_normalizados,
            modalidades_processadas=modalidades_processadas,
            total_paginas=total_paginas,
        )

        payload = {
            "metadata": {
                "url": self.url_final,
                "codigo_modalidade": self.codigo_modalidade,
                "modalidades_processadas": modalidades_processadas,
                "data_final": self.data_final,
                "tamanho_pagina": self.tamanho_pagina,
                "totalRegistros": len(dados_normalizados),
                "totalPaginas": total_paginas,
                "totalModalidades": len(resumo_modalidades),
                "ultima_atualizacao": datetime.now().isoformat(),
                "versao_schema": "3.0",
                "estrutura": "premium_plus",
            },
            "resumoModalidades": resumo_modalidades,
            "resumoPorUF": resumo_ufs,
            "resumoPorMunicipio": resumo_municipios,
            "resumoPorCategoria": resumo_categorias,
            "estatisticasGerais": estatisticas,
            "modalidades": grupos_modalidade,
        }

        return self._remover_campos_vazios(payload)

    def salvar_em_json(self, payload: Dict[str, Any], caminho_arquivo: str) -> None:
        with open(caminho_arquivo, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=4, ensure_ascii=False)

        print(f"Arquivo JSON salvo com sucesso em: {caminho_arquivo}")

    def executar(
        self,
        caminho_arquivo: str = "pncp_contratacoes_premium.json",
        modalidade_inicial: int = 1,
        modalidade_final: int = 13,
    ) -> Dict[str, Any]:
        dados_existentes = self.carregar_json_existente(caminho_arquivo)

        todos_os_dados_novos: List[Dict[str, Any]] = []
        modalidades_processadas: List[int] = []
        total_paginas = 0

        for codigo_modalidade in range(modalidade_inicial, modalidade_final + 1):
            print("\n" + "=" * 70)
            print(f"INICIANDO COLETA DA MODALIDADE {codigo_modalidade}")
            print("=" * 70)

            resultado = self.buscar_paginas_de_uma_modalidade(codigo_modalidade)

            registros = resultado["registros"]
            paginas_processadas = resultado["paginas_processadas"]

            if registros:
                todos_os_dados_novos.extend(registros)
                modalidades_processadas.append(codigo_modalidade)

            total_paginas += paginas_processadas

            print(f"FINALIZADA MODALIDADE {codigo_modalidade}")
            print(f"Registros encontrados: {len(registros)}")
            print(f"Páginas processadas: {paginas_processadas}")

        print("\n" + "=" * 70)
        print("RESUMO DA EXECUÇÃO")
        print("=" * 70)
        print(f"Registros já existentes no arquivo: {len(dados_existentes)}")
        print(f"Registros coletados nesta execução: {len(todos_os_dados_novos)}")

        dados_combinados = dados_existentes + todos_os_dados_novos

        payload = self.montar_payload_premium(
            dados_brutos=dados_combinados,
            modalidades_processadas=modalidades_processadas,
            total_paginas=total_paginas,
        )

        self.salvar_em_json(payload, caminho_arquivo)

        print("\nResumo de modalidades:")
        for item in payload.get("resumoModalidades", []):
            print(
                f'{item["modalidadeId"]} - {item["modalidadeNome"]}: '
                f'{item["quantidade"]} registro(s)'
            )

        print(f'Total de modalidades: {payload["metadata"]["totalModalidades"]}')
        print(f'Total de registros: {payload["metadata"]["totalRegistros"]}')
        print(f'Total de páginas: {payload["metadata"]["totalPaginas"]}')

        return payload

In [58]:
coletor = PncpPaginacaoParaJson(
    tamanho_pagina=50,
    data_final=20260319,
    pagina_inicial=1,
    max_paginas=None,
)

resultado = coletor.executar(
    caminho_arquivo="pncp_contratacoes_premium.json",
    modalidade_inicial=1,
    modalidade_final=13,
)

Arquivo JSON ainda não existe. Será criado um novo.

INICIANDO COLETA DA MODALIDADE 1

Consultando modalidade 1 | página 1...
Status 204 (No Content). Fim dos dados.
Nenhum registro encontrado na página 1 da modalidade 1.

Coleta finalizada para modalidade 1.
Total coletado na modalidade 1: 0
Total de páginas processadas na modalidade 1: 0
FINALIZADA MODALIDADE 1
Registros encontrados: 0
Páginas processadas: 0

INICIANDO COLETA DA MODALIDADE 2

Consultando modalidade 2 | página 1...
Status 204 (No Content). Fim dos dados.
Nenhum registro encontrado na página 1 da modalidade 2.

Coleta finalizada para modalidade 2.
Total coletado na modalidade 2: 0
Total de páginas processadas na modalidade 2: 0
FINALIZADA MODALIDADE 2
Registros encontrados: 0
Páginas processadas: 0

INICIANDO COLETA DA MODALIDADE 3

Consultando modalidade 3 | página 1...
Status 204 (No Content). Fim dos dados.
Nenhum registro encontrado na página 1 da modalidade 3.

Coleta finalizada para modalidade 3.
Total coletado n